# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [ ]:
!pip install langchain_community langchain_experimental langchain_openai
!pip install -q faiss-cpu google-search-results numexpr

In [ ]:
from typing import Optional

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [ ]:
# %pip install faiss-cpu > /dev/null
# %pip install google-search-results > /dev/null
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [ ]:
# Load the OpenAI and SerpAPI keys securely from Google Colab's Secrets manager.
# In Colab: click the key icon in the left sidebar, add secrets named exactly
# "OPENAI_API_KEY" and "SERPAPI_API_KEY", and toggle "Notebook access" ON for
# each of them for THIS notebook. No key is ever typed or hard-coded here.
#
# We also copy them into os.environ because SerpAPIWrapper() and OpenAI() read
# their keys from the environment by default rather than accepting them as
# constructor arguments in this notebook.
import os

OPENAI_API_KEY = None
SERPAPI_API_KEY = None
_source = None

try:
    from google.colab import userdata
    try:
        OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
        SERPAPI_API_KEY = userdata.get('SERPAPI_API_KEY')
        _source = "Colab secrets"
    except Exception as e:
        print(f"Could not read a Colab secret: {e}")
        print("Fix: open the key icon in the left sidebar, confirm secrets named exactly")
        print("OPENAI_API_KEY and SERPAPI_API_KEY exist, and toggle Notebook access ON for this notebook.")
except ImportError:
    # Not running in Colab - fall back to a local .env file.
    from dotenv import load_dotenv, find_dotenv
    _ = load_dotenv(find_dotenv())
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')
    _source = ".env file" if (OPENAI_API_KEY and SERPAPI_API_KEY) else None

assert OPENAI_API_KEY, (
    "OPENAI_API_KEY not found. In Colab: add it via the Secrets manager and enable "
    "notebook access for THIS notebook. Outside Colab: put it in a .env file."
)
assert SERPAPI_API_KEY, (
    "SERPAPI_API_KEY not found. In Colab: add it via the Secrets manager and enable "
    "notebook access for THIS notebook. Outside Colab: put it in a .env file."
)

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ['SERPAPI_API_KEY'] = SERPAPI_API_KEY
print(f"OPENAI_API_KEY and SERPAPI_API_KEY loaded successfully from {_source}.")

In [ ]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [ ]:
from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain.chains import LLMChain
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI

todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}"
)
todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
]


prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

/opt/anaconda3/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 0.3.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  warn_deprecated(


In [ ]:
llm = OpenAI(temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

/opt/anaconda3/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `ZeroShotAgent` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use create_react_agent instead.
  warn_deprecated(


### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [ ]:
OBJECTIVE = "Write a weather report for SF today"

In [ ]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [ ]:
baby_agi({"objective": OBJECTIVE})

/opt/anaconda3/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(



*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Thought: I should create a todo list based on the given objective
Action: TODO
Action Input: Create a todo list for SF weather report
Observation: 

1. Research the current weather patterns in San Francisco.
2. Determine the sources for obtaining accurate weather information for the city.
3. Create a list of key weather elements to include in the report, such as temperature, precipitation, wind speed, and humidity.
4. Decide on the format of the report, whether it will be a written document, a presentation, or a combination of both.
5. Set a timeline for when the report needs to be completed and shared.
6. Gather data from

{'objective': 'Write a weather report for SF today'}

---
## My Exercise: A Different Objective, a New Tool, and an Updated Agent

Per the instructions at the top ("change the objective and the associated prompts and potential tools and agents"), below I build a **second, independent BabyAGI run** rather than editing the class example above, so both are visible for comparison.

**What I changed:**
- **Objective:** instead of a weather report, the new objective is a **trip-budgeting task**: *"Plan a 3-day budget trip to Tokyo, researching top attractions and estimating total cost in USD."* This objective needs both real-world research (attractions, prices) and arithmetic (summing costs), which the original two tools (`Search`, `TODO`) can't fully do on their own.
- **New tool:** I added a **`Calculator`** tool backed by LangChain's `LLMMathChain`, so the agent can actually add up estimated costs it finds via `Search` instead of asking the LLM to eyeball arithmetic (a common source of hallucinated numbers).
- **Updated agent:** the `ZeroShotAgent`/`AgentExecutor` are rebuilt with the new 3-tool list (`Search`, `TODO`, `Calculator`) and a `prefix` that explicitly mentions budgeting, so the agent knows a running total matters for this objective.
- A **separate vectorstore** (`vectorstore_2`) is created so this second BabyAGI run doesn't share memory with the class demo above.

In [ ]:
# New tool: a Calculator backed by LLMMathChain, so the agent can reliably total
# up costs it finds via Search rather than relying on the LLM to do arithmetic
# in free text (which is a common source of small numeric hallucinations).
from langchain.chains import LLMMathChain

llm_math_chain = LLMMathChain.from_llm(llm=OpenAI(temperature=0), verbose=False)

tools_v2 = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events, prices, or attractions",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
    Tool(
        name="Calculator",
        func=llm_math_chain.run,
        description="useful for adding up estimated costs to keep a running trip budget total. Input should be a math expression, e.g. '25 + 40 + 15'.",
    ),
]

prefix_v2 = """You are an AI travel planner who performs one task based on the following objective: {objective}. \
Keep a running estimate of the total trip budget in USD as you go, using the Calculator tool to add up costs \
rather than guessing a sum yourself. Take into account these previously completed tasks: {context}."""
suffix_v2 = """Question: {task}
{agent_scratchpad}"""

prompt_v2 = ZeroShotAgent.create_prompt(
    tools_v2,
    prefix=prefix_v2,
    suffix=suffix_v2,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

llm_chain_v2 = LLMChain(llm=llm, prompt=prompt_v2)
tool_names_v2 = [tool.name for tool in tools_v2]
agent_v2 = ZeroShotAgent(llm_chain=llm_chain_v2, allowed_tools=tool_names_v2)
agent_executor_v2 = AgentExecutor.from_agent_and_tools(
    agent=agent_v2, tools=tools_v2, verbose=True
)

In [ ]:
# A separate, empty vectorstore for this second BabyAGI run, so it doesn't share
# task memory with the weather-report example above.
index_v2 = faiss.IndexFlatL2(embedding_size)
vectorstore_v2 = FAISS(embeddings_model.embed_query, index_v2, InMemoryDocstore({}), {})

In [ ]:
OBJECTIVE_V2 = "Plan a 3-day budget trip to Tokyo, researching top attractions and estimating total cost in USD."

verbose = False
max_iterations: Optional[int] = 4  # a little higher than the demo, since this objective needs research AND math

baby_agi_v2 = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore_v2,
    task_execution_chain=agent_executor_v2,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [ ]:
baby_agi_v2({"objective": OBJECTIVE_V2})

## Report: Findings from the Custom BabyAGI Run

**What I tried:** I kept the class's weather-report BabyAGI example unchanged above for reference, and built a second, independent BabyAGI instance with a new objective ("plan a 3-day budget trip to Tokyo and estimate total cost"), a third tool (`Calculator`, backed by `LLMMathChain`), an updated `prefix` that explicitly asks the agent to track a running budget total, and its own separate vectorstore memory.

**Why this objective:** the weather-report objective in the class demo only really exercises the `Search` and `TODO` tools — there's nothing in it that needs arithmetic. A trip-budgeting objective forces the agent to combine research (via `Search`, e.g. attraction/hotel/food prices) with a running numeric total (via the new `Calculator` tool), which is a more realistic stress test of whether adding a tool to BabyAGI's execution agent actually changes its behavior, not just its available action list.

**What I expect to find (to confirm once you run this with your own API keys):**
- Similar to the class demo, the task list will likely start with a `TODO`-generated plan (research attractions, research costs, draft an itinerary, sum the budget, write up the final plan) before the agent starts calling `Search`.
- The interesting behavior to watch for is whether the agent actually reaches for the `Calculator` tool when it has multiple cost figures to add, rather than just writing "$XX total" directly from its own arithmetic in the `Final Answer` — this is the same category of hallucination risk (the model doing math in free text instead of via a real calculator) that showed up in the PAL/ReAct quiz questions earlier in this project.
- With `max_iterations=4`, the run may end before a fully polished itinerary is produced (same as the weather-report demo ending after 3 iterations without a truly "finished" report) — BabyAGI's `max_iterations` is a hard cutoff, not a "wait until done" signal, so a real production version of this would need either a much higher iteration count or an explicit "is the objective complete?" check.

**What I learned:**
- Swapping BabyAGI's *objective* alone doesn't test much; the more meaningful exercise is picking an objective that stresses a **new capability** (arithmetic, in this case) so that adding a tool is not just cosmetic — the `Calculator` tool only matters here because the objective actually requires summing numbers.
- The `prefix` prompt is the main lever for nudging tool usage: explicitly telling the agent to "use the Calculator tool to add up costs rather than guessing" is a direct instruction analogous to the few-shot/instruction techniques from the earlier prompt-engineering labs — instructions can raise the odds of correct tool use, but (as seen with the PAL/ReAct/TriviaBot examples in the other labs) don't guarantee it every run.
- Giving each BabyAGI instance its own `vectorstore` matters if you want multiple independent objectives to not "bleed" completed-task memory into each other — sharing one `vectorstore` across unrelated objectives would let the second run's task creation/prioritization chains retrieve irrelevant context from the first.